In [11]:
import torch
import torch.nn as nn
from transformer import Linear, Embedding, RMSNorm, MultiHeadSelfAttention, FFN


# 问题 9：实现 Transformer 块（3 分）
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, max_seq_len: int, rope_theta: float) -> None:
        super().__init__()
        '''
        d_model: int  Transformer 块输入维度
        num_heads: int  多头自注意力头数
        d_ff: int  前馈网络中间层维度
        max_seq_len: int  RoPE 最大序列长度
        rope_theta: float  RoPE 基数频率
        '''
        # 子层 1：因果多头自注意力 + 残差连接
        self.mha = MultiHeadSelfAttention(d_model, num_heads, max_seq_len, rope_theta)
        self.pre_norm1 = RMSNorm(d_model)    # 子层 1 的前置归一化（Pre‑Norm）
        # 子层 2：前馈网络 + 残差连接
        self.ffn = FFN(d_model, d_ff)
        self.pre_norm2 = RMSNorm(d_model)    # 子层 2 的前置归一化（Pre‑Norm）

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        前置归一化（Pre‑Norm）Transformer 块：
        y = x + MHA(RMSNorm(x))
        z = y + FFN(RMSNorm(y))
        x: (batch_size, seq_len, d_model)
        return: (batch_size, seq_len, d_model)
        '''
        # part1：子层 1 = 多头自注意力 + 残差连接
        x = x + self.mha(self.pre_norm1(x))  # (B, S, d_model) -> (B, S, d_model)
        # part 2：子层 2 = 前馈网络 + 残差连接
        y = x + self.ffn(self.pre_norm2(x))  # (B, S, d_model) -> (B, S, d_model)
        return y


In [12]:
# 测试 TransformerBlock
batch_size, seq_len, d_model, num_heads, d_ff = 2, 5, 128, 8, 512
max_seq_len = 128
rope_theta = 10000.0

block = TransformerBlock(d_model=d_model, num_heads=num_heads, d_ff=d_ff, max_seq_len=max_seq_len, rope_theta=rope_theta)
x_in = torch.randn(batch_size, seq_len, d_model)
block_out = block(x_in)
print(f"Block 输入形状：{x_in.shape}，Block 输出形状：{block_out.shape}")


Block 输入形状：torch.Size([2, 5, 128])，Block 输出形状：torch.Size([2, 5, 128])


In [13]:
# 问题 10：实现 Transformer 语言模型（3 分）
class TransformerLM(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        vocab_size: int,
        context_length: int,
        num_layers: int,
        rope_theta: float
    ) -> None:
        super().__init__()
        '''
        d_model: int  Transformer 块输入维度
        num_heads: int  多头自注意力头数
        d_ff: int  前馈网络中间层维度
        vocab_size: int  词表大小，决定词元嵌入矩阵维度
        context_length: int  最大上下文长度，决定 RoPE 的 sin/cos 缓存长度
        num_layers: int  Transformer 块堆叠层数
        rope_theta: float  RoPE 基数频率
        '''
        self.embed = Embedding(vocab_size, d_model)                                  # 词元嵌入
        self.transformerBlocks = nn.Sequential(
            *[TransformerBlock(d_model, num_heads, d_ff, context_length, rope_theta=rope_theta) for _ in range(num_layers)] # 列表推导式
        )                                                                             # 多层 Transformer 块堆叠

        '''
        # 等价写法
        self.transformerBlocks = []
        
        for i in range(num_layers):
            trans_i = TransformerBlock(d_model, num_heads, d_ff, context_length, rope_theta=rope_theta)
            self.transformerBlocks.append(trans_i)    
        
        ===forward===
        # 输入 x 按顺序通过所有块的处理
        for block in self.transformerBlocks:
            x = block(x)
            
        '''
        self.final_norm = RMSNorm(d_model)                                            # 最终层归一化
        self.llm_head = Linear(d_model, vocab_size)                                   # 输出投影/预测头

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        '''
        输入: (batch_size, seq_len) 整数 token IDs
        输出: (batch_size, seq_len, vocab_size) 下一个 token 的 logits
        '''
        x_embed = self.embed(tokens)                                # (B, S) -> (B, S, d_model)
        x_blocks = self.transformerBlocks(x_embed)                 # (B, S, d_model) -> (B, S, d_model)
        llm_output = self.llm_head(self.final_norm(x_blocks))      # (B, S, d_model) -> (B, S, vocab_size)
        return llm_output


In [ ]:
n = 10
arr = [ 1 for _ in range(n)]

for x in arr:
    print(x)

In [14]:
# 测试完整 Transformer LM
batch_size, seq_len = 2, 16
d_model, num_heads, d_ff = 128, 8, 512
vocab_size, context_length, num_layers = 10000, 128, 4
rope_theta = 10000.0

model = TransformerLM(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    vocab_size=vocab_size,
    context_length=context_length,
    num_layers=num_layers,
    rope_theta=rope_theta,
)
tokens = torch.randint(0, vocab_size, (batch_size, seq_len))
logits = model(tokens)
print(f"输入 token 形状：{tokens.shape}，输出 logits 形状：{logits.shape}")


输入 token 形状：torch.Size([2, 16])，输出 logits 形状：torch.Size([2, 16, 10000])
